## Problem 0: GitHub Issues

Progress is tracked in this repo's [Issues](https://github.com/qahramantaha/computational-theory/issues): one issue per problem, commits reference issue numbers, and issues are closed when each problem is done.

## Problem 1: Representing SHA-256 Data

### 32-bit words
A word is 32 bits (section 2.1) and when you add words you do it mod 2³² (section 3.2). Python ints can be any size, so I use a normal int and mask it with `0xFFFFFFFF` to keep it at 32 bits. I put this in `add32` since I'll need it again later. The last few prints check my setup against the examples in section 3.1.

In [8]:
MASK32 = 0xFFFFFFFF   # all 32 bits on, biggest number a word can be

def add32(x, y):
    # adding two words, keeps result at 32 bits (section 3.2)
    return (x + y) & MASK32

w = 0xFFFFFFFF
print(w + 1)                          # python just keeps going past 32 bits
print(add32(w, 1))                    # with the mask it wraps back to 0
print(add32(w, 1) == (w + 1) % 2**32) # masking is the same as mod 2^32

# checking against the examples in section 3.1
print(hex(int("10100001000000111111111000100011", 2)))  # should be a103fe23
print(f"{291:08x}")                   # should be 00000123
print(f"{0x6a09e667:032b}")           # a word as 32 bits

4294967296
0
True
0xa103fe23
00000123
01101010000010011110011001100111


### Sequences of 32-bit words
I use a list of ints. It keeps the order and I can index it like `W[t]`, same as the standard does.

In [7]:
# SHA-256 initial hash value H(0), section 5.3.3
H0 = [0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a,
      0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19]

print(len(H0))
print(hex(H0[0]))

8
0x6a09e667


### Input messages
Messages are stored as `bytes`. Text gets turned into bytes with `.encode("utf-8")`, and each byte is a number from 0 to 255. The standard's own example uses "abc" (section 5.1.1), so I use the same one to check.

In [ ]:
msg = "abc".encode("utf-8")
print(msg)
print(list(msg))                       # each byte as a number
print([f"{b:08b}" for b in msg])       # should match section 5.1.1: 01100001 01100010 01100011
print(len(msg) * 8)                    # length in bits, should be 24

### 512-bit message blocks
A block is 512 bits, which is 64 bytes, and gets split into 16 words (sections 3.1 and 5.2.1). I build the padded "abc" block from section 5.1.1 by hand just to have a real block to split. Padding itself gets done properly later.

In [ ]:
# padded "abc" block from section 5.1.1: message + 1 bit + zeros + 64-bit length
block = msg + b"\x80" + bytes(52) + (len(msg) * 8).to_bytes(8, "big")
print(len(block) * 8)                  # should be 512

# split into 16 words, 4 bytes each, big-endian
words = [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]
print(len(words))                      # should be 16
print(hex(words[0]))                   # "abc" + the 1 bit -> 0x61626380
print(hex(words[15]))                  # message length 24 -> 0x18